# CloudWatch Observability를 활용한 AWS Lambda의 Amazon Bedrock AgentCore Runtime 호출

## 개요

이 자습서에서는 CloudWatch 관측성을 활성화하고 AWS Lambda 함수에서 Amazon Bedrock AgentCore Runtime에 호스팅된 Model Context Protocol(MCP) 서버 기반 Strands 에이전트를 호출하는 방법을 살펴봅니다.

### 자습서 세부 정보

| 정보                | 세부 정보                                                                        |
|:-------------------|:----------------------------------------------------------------------------------|
| 자습서 유형        | 대화형                                                                            |
| 에이전트 유형      | 단일                                                                              |
| 에이전트 프레임워크| Strands Agents                                                                    |
| LLM 모델           | Anthropic Claude Haiku 4.5                                                       |
| 자습서 구성 요소   | Lambda 호출, AgentCore Runtime, MCP 서버, CloudWatch Observability                |
| 예제 난이도        | 고급                                                                              |
| 사용 SDK           | Amazon BedrockAgentCore Python SDK, boto3, AWS Lambda                            |

### 아키텍처
```
┌─────────┐      ┌────────────────┐      ┌──────────────────┐      ┌─────────────────┐
│   API   │─────>│  AWS Lambda    │─────>│  AgentCore       │─────>│  Strands Agent  │
│  /User  │      │  (Invoker)     │      │  Runtime         │      │  + MCP Servers  │
└─────────┘      └────────────────┘      └──────────────────┘      └─────────────────┘
                        │                         │                          │
                        ▼                         ▼                          ▼
                 ┌────────────────────────────────────────────────────────────┐
                 │            CloudWatch Observability                        │
                 │       • Gen AI Traces     • Metrics     • Logs             │
                 └────────────────────────────────────────────────────────────┘
```

### 주요 기능

* 여러 MCP 서버(AWS Documentation + AWS CDK)를 Strands Agents와 통합
* Amazon Bedrock AgentCore Runtime에서 에이전트 호스팅
* AWS Lambda 함수에서 호스팅된 에이전트 호출
* CloudWatch Gen AI Observability로 에이전트 실행 모니터링
* AWS Lambda Layer for OpenTelemetry를 사용한 end-to-end 트레이스 전파

### 학습 내용

1. MCP 지원 에이전트를 AgentCore Runtime에 배포하는 방법
2. Runtime 에이전트를 호출하는 Lambda 함수를 생성하는 방법
3. 에이전트에 CloudWatch Gen AI Observability를 활성화하는 방법
4. 트레이스 전파를 위해 ADOT Lambda Layer를 구성하는 방법
5. CloudWatch console에서 트레이스와 로그를 확인하는 방법

## 사전 요구 사항

### 필수 소프트웨어
* Python 3.10+
* 적절한 권한으로 구성된 AWS 자격 증명
* Amazon Bedrock AgentCore SDK
* Lambda 함수 및 IAM role 생성 권한

### CloudWatch Transaction Search 활성화(최초 1회 설정)

**이 자습서를 시작하기 전에 CloudWatch에서 Transaction Search를 활성화해야 합니다.** AWS 계정 및 리전별로 한 번만 설정하면 됩니다.

#### Transaction Search 활성화 단계

1. **CloudWatch Console로 이동**
   - https://console.aws.amazon.com/cloudwatch/ 로 이동합니다.
   - 오른쪽 위에서 리전을 선택합니다.

2. **Gen AI Observability 열기**
   - 왼쪽 탐색 메뉴에서 **Application Signals**까지 아래로 스크롤합니다.
   - **Gen AI Observability**를 클릭합니다.

3. **Transaction Search 활성화**
   - 아직 활성화하지 않았다면 Transaction Search 활성화 배너가 표시됩니다.
   - **Enable Transaction Search**를 클릭합니다.
   - 작업을 확인합니다.

4. **설정 완료 대기**
   - Transaction Search가 완전히 작동하기까지 약 **10분**이 걸립니다.
   - 설정되는 동안 자습서를 계속 진행할 수 있습니다.

**중요 참고 사항:**
- Gen AI Observability 기능을 사용하려면 Transaction Search가 필요합니다.
- Free tier에는 스팬 1% sampling이 포함됩니다.
- 활성화하면 해당 계정/리전의 모든 AI 워크로드에 적용됩니다.

### 필수 패키지 설치

In [ ]:
!pip install --upgrade "strands-agents[otel]" strands-agents-tools boto3 bedrock-agentcore bedrock-agentcore-starter-toolkit uv

## 1단계: AWS Session 초기화 및 계정 정보 확인

In [ ]:
import boto3
import json
import time
from boto3.session import Session

# Session을 초기화하고 계정 세부 정보 확인
boto_session = Session()
region = boto_session.region_name
sts_client = boto3.client("sts")
account_id = sts_client.get_caller_identity()["Account"]

print(f"AWS Account ID: {account_id}")
print(f"Region: {region}")

## 2단계: 여러 서버를 사용하는 MCP 에이전트 생성

두 MCP 서버를 사용하는 에이전트를 생성합니다.
1. AWS Documentation MCP Server - AWS 문서에 접근
2. AWS CDK MCP Server - CDK 모범 사례 및 지침 확인

In [ ]:
%%writefile mcp_agent_multi_server.py
from strands import Agent
from strands.models import BedrockModel
from mcp import StdioServerParameters, stdio_client
from strands.tools.mcp import MCPClient
from bedrock_agentcore.runtime import BedrockAgentCoreApp

# BedrockAgentCoreApp 초기화
app = BedrockAgentCoreApp()

# AWS Documentation MCP 서버에 연결
def create_aws_docs_client():
    return MCPClient(
        lambda: stdio_client(
            StdioServerParameters(
                command="uvx", 
                args=["awslabs.aws-documentation-mcp-server@latest"]
            )
        )
    )

# AWS CDK MCP 서버에 연결
def create_cdk_client():
    return MCPClient(
        lambda: stdio_client(
            StdioServerParameters(
                command="uvx", 
                args=["awslabs.cdk-mcp-server@latest"]
            )
        )
    )

# 두 MCP 서버의 도구를 사용하는 에이전트 생성 함수
def create_agent():
    model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
    model = BedrockModel(model_id=model_id)
    
    aws_docs_client = create_aws_docs_client()
    cdk_client = create_cdk_client()
    
    with aws_docs_client, cdk_client:
        # 두 MCP 서버에서 도구 가져오기
        tools = aws_docs_client.list_tools_sync() + cdk_client.list_tools_sync()
        
        # 이 도구를 사용하는 에이전트 생성
        agent = Agent(
            model=model,
            tools=tools,
            system_prompt="""You are a helpful AWS assistant with access to AWS Documentation 
            and CDK best practices. Provide concise and accurate information about AWS services 
            and infrastructure as code patterns. When asked about pricing or CDK, use your tools 
            to search for the most current information."""
        )
    
    return agent, aws_docs_client, cdk_client

@app.entrypoint
def invoke_agent(payload):
    """입력 페이로드를 처리하고 에이전트 응답을 반환합니다."""
    agent, aws_docs_client, cdk_client = create_agent()
    
    with aws_docs_client, cdk_client:
        user_input = payload.get("prompt")
        print(f"Processing request: {user_input}")
        response = agent(user_input)
        return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

In [ ]:
%%writefile requirements.txt
strands-agents[otel]
strands-agents-tools
uv
boto3
bedrock-agentcore
aws-opentelemetry-distro==0.12.1

## 3단계: AgentCore Runtime에 에이전트 배포

MCP 에이전트를 AgentCore Runtime에 배포합니다. Runtime은 CloudWatch 관측성을 위해 OpenTelemetry로 에이전트를 자동 계측합니다.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

# Runtime 구성
agentcore_runtime = Runtime()
agent_name = "mcp_agent_lambda_observability"

# 에이전트가 이미 있는지 확인
bedrock_agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

try:
    # 기존 에이전트 목록을 조회하여 대상 에이전트가 있는지 확인
    print("Checking for existing agent...")

    # Configure가 기존 에이전트를 처리함
    config_response = agentcore_runtime.configure(
        entrypoint="mcp_agent_multi_server.py",
        auto_create_ecr=True,
        auto_create_execution_role=True,
        requirements_file="requirements.txt",
        region=region,
        agent_name=agent_name,
    )

    print(f"✅ Agent configured: {agent_name}")

except Exception as e:
    print(f"Configuration note: {e}")
    print("Continuing with existing configuration...")

In [ ]:
# AgentCore Runtime에서 에이전트 시작 또는 업데이트
print("🚀 Deploying/Updating agent to AgentCore Runtime (this may take several minutes)...")

try:
    # 멱등성 있는 배포를 위해 auto_update_on_conflict=True 사용
    launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)

    print("\n✅ Agent deployed successfully!")
    print(f"Agent ARN: {launch_result.agent_arn}")
    print(f"Agent ID: {launch_result.agent_id}")

    # 에이전트 세부 정보 저장
    agent_arn = launch_result.agent_arn
    agent_id = launch_result.agent_id

except Exception as e:
    # 대체 처리: 시작에 실패하면 기존 에이전트 상태 조회
    if "already exists" in str(e).lower() or "conflict" in str(e).lower():
        print("Agent already exists, retrieving existing agent details...")
        try:
            status_response = agentcore_runtime.status()

            # 에이전트 정보를 안전하게 추출
            if status_response and hasattr(status_response, "endpoint") and status_response.endpoint:
                agent_arn = status_response.endpoint.get("agentArn", "")
                if not agent_arn:
                    # 다른 필드 이름 시도
                    agent_arn = status_response.endpoint.get("agentRuntimeArn", "")
                agent_id = agent_name
                print(f"✅ Using existing agent: {agent_arn}")
            else:
                # 상태에 엔드포인트 정보가 없으면 agent_name 사용
                print("⚠️ Could not retrieve full agent details, using agent name")
                agent_id = agent_name
                agent_arn = ""  # 다음 단계에서 입력됨

        except Exception as status_error:
            print(f"Status retrieval error: {status_error}")
            print("Using agent name as fallback")
            agent_id = agent_name
            agent_arn = ""
    else:
        print(f"Deployment error: {e}")
        raise e

In [ ]:
# 에이전트가 준비될 때까지 대기
print("⏳ Waiting for agent endpoint to be ready...")
status_response = agentcore_runtime.status()
status = status_response.endpoint.get("status", "UNKNOWN")
end_statuses = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]

max_attempts = 60
attempt = 0

while status not in end_statuses and attempt < max_attempts:
    time.sleep(10)
    try:
        status_response = agentcore_runtime.status()
        status = status_response.endpoint.get("status", "UNKNOWN")
        print(f"Status: {status}")
    except Exception as e:
        print(f"Status check error: {e}")
        break
    attempt += 1

if status == "READY":
    print("\n✅ Agent is ready to accept invocations!")
else:
    print(f"\n⚠️ Agent status: {status}")

# Runtime ARN 가져오기(Lambda에 필요)
agent_runtime_arn = status_response.endpoint.get("agentRuntimeArn", "")
if not agent_runtime_arn:
    agent_runtime_arn = status_response.endpoint.get("endpointArn", "")

print(f"Runtime ARN: {agent_runtime_arn}")

## 4단계: 직접 호출 테스트

먼저 에이전트를 직접 테스트하여 올바르게 작동하는지 확인합니다.

In [ ]:
# 직접 호출 테스트
test_payload = {"prompt": "What is Amazon Bedrock's pricing model?"}

print(f"Testing agent with prompt: {test_payload['prompt']}\n")

try:
    invoke_response = agentcore_runtime.invoke(test_payload)

    # 응답 표시
    from IPython.display import Markdown, display

    response_text = invoke_response["response"][0]
    display(Markdown(response_text))
except Exception as e:
    print(f"Direct invocation test error: {e}")
    print("This may be normal if the agent is still initializing. Continue to next step.")

## 5단계: Lambda 실행 역할 생성

Lambda 함수에는 다음 권한이 필요합니다.
1. AgentCore Runtime 에이전트 호출
2. CloudWatch에 로그 기록
3. X-Ray로 트레이스 전송
4. 향상된 관측성을 위해 Application Signals 사용

In [ ]:
iam_client = boto3.client("iam")

# Lambda execution role 이름 정의
lambda_role_name = f"AgentCoreLambdaExecutionRole-{agent_name}"

# Lambda용 trust policy
lambda_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

# Role 생성 또는 가져오기
try:
    role_response = iam_client.create_role(
        RoleName=lambda_role_name,
        AssumeRolePolicyDocument=json.dumps(lambda_trust_policy),
        Description="Execution role for Lambda to invoke AgentCore Runtime with observability",
    )
    lambda_role_arn = role_response["Role"]["Arn"]
    print(f"✅ Created new Lambda execution role: {lambda_role_arn}")
    time.sleep(10)  # Role 전파 대기
except iam_client.exceptions.EntityAlreadyExistsException:
    role_response = iam_client.get_role(RoleName=lambda_role_name)
    lambda_role_arn = role_response["Role"]["Arn"]
    print(f"✅ Using existing Lambda execution role: {lambda_role_arn}")

# AWS managed policy 연결
managed_policies = [
    "arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",  # CloudWatch Logs
    "arn:aws:iam::aws:policy/AWSXRayDaemonWriteAccess",  # X-Ray tracing
    "arn:aws:iam::aws:policy/CloudWatchLambdaApplicationSignalsExecutionRolePolicy",  # Application Signals
]

for policy_arn in managed_policies:
    try:
        iam_client.attach_role_policy(RoleName=lambda_role_name, PolicyArn=policy_arn)
        print(f"✅ Attached policy: {policy_arn.split('/')[-1]}")
    except iam_client.exceptions.InvalidInputException:
        print(f"⚠️ Policy already attached: {policy_arn.split('/')[-1]}")
    except Exception as e:
        if "already attached" in str(e).lower():
            print(f"⚠️ Policy already attached: {policy_arn.split('/')[-1]}")
        else:
            print(f"⚠️ Policy attachment note: {e}")

In [ ]:
# AgentCore Runtime 호출용 custom policy 생성
agentcore_policy_name = f"AgentCoreRuntimeInvokePolicy-{agent_name}"

# 이 에이전트에만 적용되는 범위가 축소된 policy
agentcore_policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AgentCoreRuntimeAccess",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:InvokeAgentRuntime"],
            "Resource": agent_runtime_arn,  # 이 runtime으로만 제한
        },
        {
            "Sid": "BedrockModelAccess",
            "Effect": "Allow",
            "Action": ["bedrock:InvokeModel"],
            "Resource": f"arn:aws:bedrock:{region}::foundation-model/*",  # Bedrock 모델
        },
    ],
}

try:
    policy_response = iam_client.create_policy(
        PolicyName=agentcore_policy_name,
        PolicyDocument=json.dumps(agentcore_policy_document),
        Description="Scoped policy to allow Lambda to invoke specific AgentCore Runtime",
    )
    agentcore_policy_arn = policy_response["Policy"]["Arn"]
    print(f"✅ Created AgentCore invocation policy: {agentcore_policy_arn}")
except iam_client.exceptions.EntityAlreadyExistsException:
    agentcore_policy_arn = f"arn:aws:iam::{account_id}:policy/{agentcore_policy_name}"
    print(f"✅ Using existing AgentCore invocation policy: {agentcore_policy_arn}")

    # 기존 policy document 업데이트
    try:
        # 기존 policy version 가져오기
        versions = iam_client.list_policy_versions(PolicyArn=agentcore_policy_arn)["Versions"]

        # Version이 5개이면 가장 오래된 non-default version 삭제
        if len(versions) >= 5:
            oldest_version = sorted(
                [v for v in versions if not v["IsDefaultVersion"]],
                key=lambda x: x["CreateDate"],
            )[0]
            iam_client.delete_policy_version(PolicyArn=agentcore_policy_arn, VersionId=oldest_version["VersionId"])

        # 새 version 생성
        iam_client.create_policy_version(
            PolicyArn=agentcore_policy_arn,
            PolicyDocument=json.dumps(agentcore_policy_document),
            SetAsDefault=True,
        )
        print("  ✅ Updated policy to latest runtime ARN")
    except Exception as e:
        print(f"  ℹ️ Policy update note: {e}")

# Custom policy를 role에 연결
try:
    iam_client.attach_role_policy(RoleName=lambda_role_name, PolicyArn=agentcore_policy_arn)
    print("✅ Attached AgentCore invocation policy to Lambda role")
except Exception as e:
    if "already attached" in str(e).lower():
        print("⚠️ Policy already attached")
    else:
        print(f"⚠️ Policy attachment note: {e}")

print("\n✅ Lambda role configured with minimal required permissions")

## 6단계: AWS Lambda Layer for OpenTelemetry 구성

Lambda에서 AgentCore Runtime까지 전체 트레이스를 전파하려면 AWS Lambda Layer for OpenTelemetry(ADOT)를 추가해야 합니다. 이 Layer는 다음 기능을 제공합니다.

- OpenTelemetry로 Lambda 함수 자동 계측
- Downstream 서비스(AgentCore Runtime)로 트레이스 컨텍스트 전파
- CloudWatch 트레이스에서 end-to-end 가시성 제공

**이 Layer가 없으면 Lambda와 AgentCore Runtime 간 트레이스가 서로 연결되지 않습니다.**

In [ ]:
# 리전별 ADOT Lambda Layer ARN(Python)
# 최신 ARN: https://aws-otel.github.io/docs/getting-started/lambda/lambda-python
adot_layer_arns = {
    "us-east-1": "arn:aws:lambda:us-east-1:615299751070:layer:AWSOpenTelemetryDistroPython:18",
    "us-east-2": "arn:aws:lambda:us-east-2:615299751070:layer:AWSOpenTelemetryDistroPython:15",
    "us-west-1": "arn:aws:lambda:us-west-1:615299751070:layer:AWSOpenTelemetryDistroPython:22",
    "us-west-2": "arn:aws:lambda:us-west-2:615299751070:layer:AWSOpenTelemetryDistroPython:22",
    "ap-south-1": "arn:aws:lambda:ap-south-1:615299751070:layer:AWSOpenTelemetryDistroPython:15",
    "ap-northeast-2": "arn:aws:lambda:ap-northeast-2:615299751070:layer:AWSOpenTelemetryDistroPython:15",
    "ap-southeast-1": "arn:aws:lambda:ap-southeast-1:615299751070:layer:AWSOpenTelemetryDistroPython:14",
    "ap-southeast-2": "arn:aws:lambda:ap-southeast-2:615299751070:layer:AWSOpenTelemetryDistroPython:15",
    "ap-northeast-1": "arn:aws:lambda:ap-northeast-1:615299751070:layer:AWSOpenTelemetryDistroPython:15",
    "eu-central-1": "arn:aws:lambda:eu-central-1:615299751070:layer:AWSOpenTelemetryDistroPython:15",
    "eu-west-1": "arn:aws:lambda:eu-west-1:615299751070:layer:AWSOpenTelemetryDistroPython:15",
    "eu-west-2": "arn:aws:lambda:eu-west-2:615299751070:layer:AWSOpenTelemetryDistroPython:15",
}

adot_layer_arn = adot_layer_arns.get(region)
if not adot_layer_arn:
    print(f"⚠️ Warning: ADOT Layer ARN not defined for region {region}")
    print("Please check https://aws-otel.github.io/docs/getting-started/lambda/lambda-python for the latest ARN")
    print("Continuing without ADOT Layer - trace propagation may be limited")
else:
    print(f"✅ Using ADOT Layer for region {region}:")
    print(f"   {adot_layer_arn}")

## 7단계: Lambda 함수 생성

이 Lambda 함수는 X-Ray tracing과 ADOT 계측이 활성화된 AgentCore Runtime 에이전트를 호출합니다.

In [ ]:
%%writefile lambda_agentcore_invoker.py
import json
import boto3
import os
import traceback
from botocore.exceptions import ClientError

def lambda_handler(event, context):
    """
    AgentCore Runtime 에이전트를 호출하는 Lambda 함수입니다.
    
    예상 이벤트 형식:
    {
        "prompt": "Your question here",
        "sessionId": "optional-session-id"
    }
    """
    
    # boto3 client 초기화
    bedrock_agentcore_client = boto3.client('bedrock-agentcore')
    
    try:
        # 환경 변수 가져오기
        runtime_arn = os.environ.get('RUNTIME_ARN')
        
        print(f"Lambda function started")
        print(f"Runtime ARN: {runtime_arn}")
        
        if not runtime_arn:
            return {
                'statusCode': 500,
                'body': json.dumps({
                    'error': 'Configuration Error',
                    'message': 'Missing RUNTIME_ARN environment variable'
                })
            }
        
        # 입력 parsing
        if isinstance(event, str):
            event = json.loads(event)
        
        prompt = event.get('prompt', '')
        session_id = event.get('sessionId', context.aws_request_id)
        
        if not prompt:
            return {
                'statusCode': 400,
                'body': json.dumps({
                    'error': 'Bad Request',
                    'message': 'Missing prompt in request'
                })
            }
        
        print(f"Processing prompt: {prompt}")
        print(f"Session ID: {session_id}")
        
        # AgentCore용 페이로드 준비
        payload = json.dumps({"prompt": prompt})
        
        # AgentCore Runtime 호출
        print("Invoking AgentCore Runtime...")
        response = bedrock_agentcore_client.invoke_agent_runtime(
            agentRuntimeArn=runtime_arn,
            runtimeSessionId=session_id,
            payload=payload
        )
        
        print("Response received from AgentCore")
        
        # 응답 parsing - StreamingBody 처리
        agent_response = None
        
        if 'response' in response:
            response_body = response['response']
            
            # StreamingBody 처리
            if hasattr(response_body, 'read'):
                raw_data = response_body.read()
                if isinstance(raw_data, bytes):
                    agent_response = raw_data.decode('utf-8')
                else:
                    agent_response = str(raw_data)
            elif isinstance(response_body, list) and len(response_body) > 0:
                if isinstance(response_body[0], bytes):
                    agent_response = response_body[0].decode('utf-8')
                else:
                    agent_response = str(response_body[0])
            elif isinstance(response_body, bytes):
                agent_response = response_body.decode('utf-8')
            elif isinstance(response_body, str):
                agent_response = response_body
            else:
                agent_response = str(response_body)
        
        if not agent_response:
            agent_response = "No response from agent"
            print("Warning: No response extracted from AgentCore")
        
        print(f"Agent response received (length: {len(agent_response)} chars)")
        
        return {
            'statusCode': 200,
            'body': json.dumps({
                'response': agent_response,
                'sessionId': session_id
            }),
            'headers': {
                'Content-Type': 'application/json'
            }
        }
        
    except ClientError as e:
        error_code = e.response['Error']['Code']
        error_message = e.response['Error']['Message']
        print(f"AWS ClientError: {error_code}")
        print(f"Error message: {error_message}")
        traceback.print_exc()
        
        return {
            'statusCode': 500,
            'body': json.dumps({
                'error': error_code,
                'message': error_message
            })
        }
    
    except Exception as e:
        print(f"Unexpected error: {str(e)}")
        print(f"Error type: {type(e).__name__}")
        traceback.print_exc()
        
        return {
            'statusCode': 500,
            'body': json.dumps({
                'error': 'InternalError',
                'message': str(e),
                'type': type(e).__name__
            })
        }

In [ ]:
# 배포 패키지 생성
import zipfile
from pathlib import Path

zip_filename = "lambda_agentcore_invoker.zip"
lambda_file = "lambda_agentcore_invoker.py"

print("Creating Lambda deployment package...")

# 파일 존재 여부 확인
if not Path(lambda_file).exists():
    raise FileNotFoundError(f"{lambda_file} not found. Run the previous cell first.")

# 기존 zip 파일이 있으면 삭제
if Path(zip_filename).exists():
    Path(zip_filename).unlink()
    print(f"Removed old {zip_filename}")

# 새 zip 파일 생성
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(lambda_file, arcname=lambda_file)

print(f"✅ Created Lambda deployment package: {zip_filename}")

# Zip 파일을 byte로 읽기
with open(zip_filename, "rb") as f:
    lambda_zip_content = f.read()

print(f"Package size: {len(lambda_zip_content):,} bytes")

In [ ]:
# ADOT Layer 및 X-Ray tracing을 사용하는 Lambda 함수 생성 또는 업데이트
lambda_client = boto3.client("lambda", region_name=region)
lambda_function_name = f"agentcore-mcp-invoker-{agent_name}"

print("📋 Lambda Configuration:")
print(f"  Function Name: {lambda_function_name}")
print(f"  Runtime ARN:   {agent_runtime_arn}")
print(f"  ADOT Layer:    {adot_layer_arn if adot_layer_arn else 'Not configured'}")
print(f"  AWS Region:    {region}")
print()

lambda_config = {
    "FunctionName": lambda_function_name,
    "Runtime": "python3.12",
    "Role": lambda_role_arn,
    "Handler": "lambda_agentcore_invoker.lambda_handler",
    "Code": {"ZipFile": lambda_zip_content},
    "Description": "Lambda function to invoke AgentCore Runtime with MCP servers and ADOT instrumentation",
    "Timeout": 300,
    "MemorySize": 512,
    "Environment": {
        "Variables": {
            "RUNTIME_ARN": agent_runtime_arn,
            "AWS_LAMBDA_EXEC_WRAPPER": "/opt/otel-instrument",  # ADOT 자동 계측 활성화
        }
    },
    "TracingConfig": {
        "Mode": "Active"  # X-Ray tracing 활성화
    },
}

# 사용 가능한 경우 ADOT Layer 추가
if adot_layer_arn:
    lambda_config["Layers"] = [adot_layer_arn]

try:
    lambda_response = lambda_client.create_function(**lambda_config)
    lambda_function_arn = lambda_response["FunctionArn"]
    print(f"✅ Created Lambda function: {lambda_function_name}")
    print(f"Function ARN: {lambda_function_arn}")
except lambda_client.exceptions.ResourceConflictException:
    print("⚠️  Function exists, updating...")

    # 코드 업데이트
    lambda_client.update_function_code(FunctionName=lambda_function_name, ZipFile=lambda_zip_content)
    print("  ✅ Code updated")

    # 코드 업데이트가 처리될 때까지 잠시 대기
    time.sleep(2)

    # 구성 업데이트
    update_config = {
        "FunctionName": lambda_function_name,
        "Environment": lambda_config["Environment"],
        "TracingConfig": lambda_config["TracingConfig"],
        "Timeout": lambda_config["Timeout"],
        "MemorySize": lambda_config["MemorySize"],
    }

    # 사용 가능한 경우 layer 추가
    if adot_layer_arn:
        update_config["Layers"] = [adot_layer_arn]

    lambda_client.update_function_configuration(**update_config)
    print("  ✅ Configuration updated")

    lambda_function_arn = lambda_client.get_function(FunctionName=lambda_function_name)["Configuration"]["FunctionArn"]
    print(f"✅ Updated existing Lambda function: {lambda_function_name}")
except Exception as e:
    print(f"Lambda creation/update error: {e}")
    raise e

# 함수가 준비될 때까지 대기
print("\n⏳ Waiting for Lambda function to be active...")
try:
    waiter = lambda_client.get_waiter("function_active_v2")
    waiter.wait(FunctionName=lambda_function_name)
    print("✅ Lambda function is active and ready!")
except Exception as e:
    print(f"Waiter note: {e}")
    print("Continuing...")

print("\n✅ X-Ray Active Tracing is ENABLED")
if adot_layer_arn:
    print("✅ ADOT Layer is ENABLED - Trace context will propagate to AgentCore Runtime")
else:
    print("⚠️ ADOT Layer is NOT enabled - Trace propagation may be limited")

## 8단계: Lambda 함수 호출 및 테스트

Lambda 함수를 호출하여 통합을 테스트합니다.

In [ ]:
# 테스트 페이로드
test_payloads = [
    {"prompt": "What is AWS Lambda? Answer in one sentence."},
    {"prompt": "Name 2 AWS compute services."},
    {"prompt": "What is Amazon S3 used for? Be brief."},
]

print("🚀 Invoking Lambda function with test payloads...\n")

for i, payload in enumerate(test_payloads, 1):
    print(f"\n{'=' * 80}")
    print(f"Test {i}: {payload['prompt']}")
    print("=" * 80)

    try:
        response = lambda_client.invoke(
            FunctionName=lambda_function_name,
            InvocationType="RequestResponse",
            Payload=json.dumps(payload),
        )

        response_payload = json.loads(response["Payload"].read())

        if "FunctionError" in response:
            print("\n❌ Lambda Function Error!")
            print(f"Error Type: {response_payload.get('errorType', 'Unknown')}")
            print(f"Error Message: {response_payload.get('errorMessage', 'Unknown')}")
            continue

        if response_payload["statusCode"] == 200:
            body = json.loads(response_payload["body"])
            print("\n✅ Success!")
            print(f"Session ID: {body.get('sessionId', 'N/A')}")

            response_text = body.get("response", "")
            print(f"\nAgent Response:\n{response_text[:300]}...") if len(response_text) > 300 else print(
                f"\nAgent Response:\n{response_text}"
            )
        else:
            print(f"\n❌ Error Response (Status: {response_payload['statusCode']})")
            body = json.loads(response_payload["body"])
            print(f"Error: {body.get('error', 'Unknown')}")
            print(f"Message: {body.get('message', 'Unknown')}")

    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        import traceback

        traceback.print_exc()

    time.sleep(2)

print("\n" + "=" * 80)
print("✅ All test invocations completed!")
print("\n⏰ Traces are being processed and will be available in CloudWatch within 1-2 minutes.")

## 9단계: CloudWatch에서 Observability 데이터 확인

ADOT 계측으로 트레이스를 생성했으므로 이제 CloudWatch console에서 Lambda부터 AgentCore Runtime까지 연결된 전체 end-to-end 트레이스를 확인할 수 있습니다.

### CloudWatch Observability 대시보드

#### 1. CloudWatch Gen AI Observability 대시보드
![image.png](./image/image1.png)
에이전트 성능을 확인하는 기본 대시보드입니다.
- **기능**: Agents View, Sessions View, 스팬 타임라인을 제공하는 Traces
- **지표**: 토큰 사용량, 소요 시간, 오류율, 도구 호출
- **트레이스 연속성**: ADOT Layer를 사용하면 Lambda → AgentCore Runtime으로 연결된 트레이스를 확인할 수 있습니다.

![image2.png](./image/image2.png)

#### 2. CloudWatch Logs
원본 실행 로그입니다.
- **Lambda Logs**: 함수 실행 세부 정보
- **AgentCore Logs**: 에이전트 처리 단계 및 MCP 서버 통신

### Console 바로 가기

In [ ]:
# CloudWatch console URL 생성
base_url = f"https://{region}.console.aws.amazon.com/cloudwatch"

urls = {
    "Gen AI Observability Dashboard": f"{base_url}/home?region={region}#/gen-ai-observability/agent-core/agents",
    "Lambda Function Logs": f"{base_url}/home?region={region}#logsV2:log-groups/log-group/$252Faws$252Flambda$252F{lambda_function_name}",
    "AgentCore Runtime Logs": f"{base_url}/home?region={region}#logsV2:log-groups/log-group/$252Faws$252Fbedrock-agentcore$252Fruntimes$252F{agent_id}-DEFAULT",
}

print("📊 CloudWatch Observability Links\n")
for name, url in urls.items():
    print(f"{name}:")
    print(f"  {url}\n")

print("\n📊 Key Observability Features:")
print("\n1. Gen AI Observability Dashboard:")
print("   • Session duration, count, and conversation flow")
print("   • Token usage (input/output) and costs")
print("   • MCP tool invocation traces")
print("   • Error rates and latency metrics")
print("   • End-to-end trace from Lambda to AgentCore (with ADOT Layer)")
print("\n2. CloudWatch Logs:")
print("   • Lambda execution logs with request/response data")
print("   • Agent processing steps and decision-making")
print("   • MCP server communication logs")
print("   • OpenTelemetry instrumentation data")

print("\n💡 How to view logs in CloudWatch Console:")
print("\n  Lambda Logs:")
print("   1. Click the 'Lambda Function Logs' link above")
print("   2. Select the most recent log stream")
print("   3. View execution details including:")
print("      • START/END RequestId markers")
print("      • Processing prompt messages")
print("      • Response received confirmations")
print("      • Duration and memory usage in REPORT")
print("\n  AgentCore Logs:")
print("   1. Click the 'AgentCore Runtime Logs' link above")
print("   2. Look for 'runtime-logs' log stream")
print("   3. View agent execution including:")
print("      • MCP tool invocations")
print("      • Agent reasoning steps")
print("      • OpenTelemetry trace data")

if adot_layer_arn:
    print("\n✅ ADOT Layer Enabled - Trace Context Propagation:")
    print("   • Traces will show Lambda → AgentCore Runtime connection")
    print("   • View complete request flow in CloudWatch X-Ray trace map")
    print("   • Session IDs and trace IDs are propagated across services")
else:
    print("\n⚠️ ADOT Layer Not Enabled:")
    print("   • Traces may be disconnected between Lambda and AgentCore")
    print("   • Consider adding ADOT Layer for complete observability")

## 요약 및 모범 사례

### 완료한 작업

✅ **에이전트 배포**: AWS Docs + CDK 서버를 사용하는 MCP 에이전트를 생성하여 AgentCore Runtime에 배포

✅ **Lambda 통합**: 호스팅된 에이전트를 호출하는 Lambda 함수 구축

✅ **ADOT 계측**: 트레이스 전파를 위해 AWS Lambda Layer for OpenTelemetry 추가

✅ **CloudWatch Observability**: 모니터링을 위해 Gen AI Observability 활성화

✅ **End-to-End 테스트**: Lambda → AgentCore → MCP 흐름을 보여 주는 트레이스 생성

### 주요 Observability 구성 요소

1. **ADOT Lambda Layer**: Lambda에서 downstream 서비스로 트레이스 컨텍스트 전파
2. **X-Ray Active Tracing**: 요청 경로 시각화
3. **Gen AI Traces**: 도구 호출이 포함된 전체 스팬 타임라인
4. **CloudWatch Logs**: Timestamp가 포함된 상세 실행 로그
5. **성능 지표**: 소요 시간, 토큰 사용량, 오류율

### 모범 사례

1. **항상 ADOT Layer 사용**: 서비스 경계를 넘어 트레이스 연속성 보장
2. **Session ID**: 일관된 session ID를 사용하여 대화 추적
3. **지표 모니터링**: 소요 시간, 토큰 사용량, 오류율 추적
4. **Alarm 설정**: 임계값에 대한 CloudWatch alarm 생성
5. **로그 보존**: 적절한 보존 기간 구성
6. **정기 검토**: 트레이스를 분석하여 병목 구간 식별

### 다음 단계

1. **대시보드 살펴보기**: CloudWatch Gen AI Observability 방문
2. **트레이스 분석**: 개별 트레이스 타임라인을 검토하고 Lambda → AgentCore 연결 확인
3. **Alarm 생성**: 오류율 및 지연 시간 alert 설정
4. **최적화**: 트레이스 데이터를 사용하여 성능 개선
5. **확장**: 트래픽 증가에 따라 모니터링하고 조정

### 리소스 정리

In [ ]:
# # 리소스를 삭제하려면 주석 해제

# # Lambda 함수 삭제
# try:
#     lambda_client.delete_function(FunctionName=lambda_function_name)
#     print(f"✅ Deleted Lambda function: {lambda_function_name}")
# except Exception as e:
#     print(f"Lambda deletion note: {e}")

# # IAM policy 연결 해제 및 삭제
# for policy_arn in managed_policies + [agentcore_policy_arn]:
#     try:
#         iam_client.detach_role_policy(RoleName=lambda_role_name, PolicyArn=policy_arn)
#     except Exception as e:
#         print(f"Policy detach note: {e}")

# # IAM role 삭제
# try:
#     iam_client.delete_role(RoleName=lambda_role_name)
#     print(f"✅ Deleted IAM role: {lambda_role_name}")
# except Exception as e:
#     print(f"Role deletion note: {e}")

# # Custom policy 삭제
# try:
#     iam_client.delete_policy(PolicyArn=agentcore_policy_arn)
#     print(f"✅ Deleted custom policy: {agentcore_policy_name}")
# except Exception as e:
#     print(f"Policy deletion note: {e}")

# # AgentCore Runtime 삭제
# try:
#     agentcore_runtime.delete()
#     print(f"✅ Deleted AgentCore Runtime agent: {agent_name}")
# except Exception as e:
#     print(f"AgentCore deletion note: {e}")

print("To delete resources, uncomment the code above and run this cell.")

## 축하합니다! 🎉

다음 작업을 성공적으로 완료했습니다.
- MCP 지원 에이전트를 Amazon Bedrock AgentCore Runtime에 배포
- 트레이스 전파를 위한 ADOT Layer가 포함된 Lambda 함수 생성
- CloudWatch Gen AI Observability 구성
- CloudWatch console에서 end-to-end 트레이스 생성 및 확인

이제 전체 관측성을 갖춘 production-ready AI 에이전트를 구축할 수 있습니다.

### 추가 자료

- [Amazon Bedrock AgentCore 문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/what-is-bedrock-agentcore.html)
- [CloudWatch Gen AI Observability 가이드](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/GenAI-observability.html)
- [AWS Lambda Layer for OpenTelemetry](https://aws-otel.github.io/docs/getting-started/lambda)
- [AWS X-Ray 문서](https://docs.aws.amazon.com/xray/latest/devguide/)
- [Model Context Protocol (MCP)](https://modelcontextprotocol.io/)